In [ ]:
# Demo: Leveraging metaclasses in a custom ORM

# Key components of an ORM:
# Model classes - represent database tables
# Mapping - define how model classes map to database tables
# Database connection - manage the connection to the database
# Querying - provide a way to query the database using model classes

# Metaclasses
# Architects of your classes - control how your classes are created and behave

# Metaclasses

In [1]:
class ModelMetaclass(type):
    """A meta class for creating ORM Model classes."""
    def __new__(cls, name, bases, attrs):
        # Get the table name from the class
        table_name = attrs.get('__tablename__', name.lower())
        # Add the table name for the class attributes
        attrs['__tablename__'] = table_name
        # Create the new class
        return super().__new__(cls, name, bases, attrs)
    # NOTE: you must use underscores for __new__ because 
    # it is a special method in Python that is called when a new instance of a class is created. 
    # The double underscores indicate that it is a special method and not a regular method.

class Model(metaclass=ModelMetaclass):
    """Base class for all ORM models."""
    def __init__(self, **kwargs):
        for key, value in kwargs.items():
            setattr(self, key, value)

    def save(self):
        """Save the model instance to the database."""
        # Here you would implement the logic to save the instance to the database
        print(f"Saving {self.__class__.__name__} to table {self.__tablename__} with data: {self.__dict__}")

class User(Model):
    """User model representing the users table."""
    __tablename__ = 'users'

    def __init__(self, username, email):
        super().__init__(username=username, email=email)

print(User.__tablename__)  # Output: users


users


# Building ORMs Using Metaclasses

In [2]:
class ORMMetaclass(type):
    """A meta class for creating ORM Model classes."""
    def __new__(cls, name, bases, attrs):
        if name == 'Model':
            return super().__new__(cls, name, bases, attrs)
        table_name = attrs.get('__tablename__', name.lower())
        mappings = {}
        for k, v, in attrs.items():
            if isinstance(v, Field):
                mappings[k] = v
        for k in mappings.keys():
            attrs.pop(k)

        attrs['__mappings__'] = mappings
        attrs['__table__'] = table_name
        return super().__new__(cls, name, bases, attrs)

class Field(object):
    """A class representing a database field."""
    def __init__(self, column_type):
        self.column_type = column_type

class Stringfield(Field):
    """A class representing a string field in the database."""
    def __init__(self, column_type='varchar(255)'):
        super().__init__(column_type)

class Model(metaclass=ORMMetaclass):
    """Base class for all ORM models."""
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)

    def __str__(self):
        return f"<{self.__class__.__name__} {self.id}>"

class User(Model):
    """User model representing the users table."""
    __tablename__ = 'users'

    id = Stringfield('INT PRIMARY KEY AUTO_INCREMENT')
    username = Stringfield()
    email = Stringfield()

user = User(name="John Doe", email="john.doe@example.com")
print(user.__mappings__)
print(user.__table__)  # Output: users


{'id': <__main__.Stringfield object at 0x111438580>, 'username': <__main__.Stringfield object at 0x111438610>, 'email': <__main__.Stringfield object at 0x1114385e0>}
users


In [ ]:
# Benefits of Using Metaclasses in an ORM:
# 1. Reduced Boilerplate Code: Metaclasses can automatically 
# generate common methods and attributes for your model classes, 
# reducing the amount of boilerplate code you need to write.

# 2. Improved code organization: Metaclasses can help organize your code 
# by separating the logic for creating and managing model classes from the logic 
# for defining the model classes themselves.

# 3. Increased flexibility: Metaclasses can provide a high degree of flexibility 
# in how your model classes are defined and behave, allowing you to create more 
# complex and powerful ORM systems.

# Key takeaways:
# Metaclasses -
# Control class creation
# Modify class behavior without changing the class definition directly

# ORMs
# Simplify database interactions
# Map database tables to Python classes
# Make database operations more intuitive and Pythonic